# 2.9f - Théorie effective du grokking en Lean : apprendre, extraire, prouver

**Navigation** : [<< 2.9e - MIPS (extraction de programme)](2.9e-MIPS-Extraction-Programme.ipynb) · Lake : [`learning_theory_lean`](../../learning_theory_lean/README.md) · [2.10 - Optimisation et hyperparamètres >>](2.10-Optimisation-Hyperparametres.ipynb)

Ce carnet est le **compagnon kernel Lean** (`lean4-wsl`) du module `EffectiveTheory`
du lake `learning_theory_lean`, au sens de l'EPIC #11703. Il ferme la chaîne que la
famille 2.9 parcourt en Python :

| Jambe | Où elle se fait | Ce qu'elle produit |
|---|---|---|
| **Apprendre** | [2.9](2.9-Grokking-Generalisation.ipynb), [2.9c](2.9c-Grokking-Diagrammes-Phases.ipynb) | un réseau qui grokke ; la carte des quatre phases |
| **Extraire** | [2.9d](2.9d-Features-Circulaires-Helice-Nombres.ipynb), [2.9e](2.9e-MIPS-Extraction-Programme.ipynb) | la géométrie latente ; le programme sorti du réseau |
| **Prouver** | **ce carnet** + [`EffectiveTheory/Grokking.lean`](../../learning_theory_lean/EffectiveTheory/Grokking.lean) | les mêmes énoncés, cette fois **démontrés** |

Le papier de référence est **R02** - Liu, Michaud & Tegmark, *Towards Understanding
Grokking: An Effective Theory of Representation Learning* (arXiv:2205.10343, 2022).
Sa thèse tient en une phrase : à perte d'entraînement nulle, la représentation
apprise doit respecter l'arithmétique de la tâche, et cette contrainte prend la
forme de **parallélogrammes** dans l'espace des embeddings.

**Ce que ce carnet n'est pas.** Les deux premières jambes ne sont pas rejouées ici :
elles appartiennent aux carnets Python de la famille, qui tournent sur PyTorch. Ce
carnet prend l'**organe natif** - le lake - et l'interroge : les énoncés que la
série manipule numériquement y sont des théorèmes. Aucune réimplémentation.

**Plan.**

| # | Section | Objet |
|---|---|---|
| 1 | Le lake répond | `import` réel, `#check` des déclarations sous leurs vrais noms |
| 2 | Apprendre | la perte effective `loss0` du lake, rendue **calculable**, et sa descente de gradient |
| 3 | Extraire | le résidu de parallélogramme, mesuré sur l'embedding entraîné |
| 4 | Prouver | les théorèmes du lake, **instanciés** - pas seulement affichés |
| 5 | Conserver | les lois de conservation de l'App. F, et leur témoin numérique |
| 6 | Limites | ce que le lake ne prouve **pas** |
| 7 | Exercices | trois énoncés à compléter |

## 1. Le lake répond

Le lake `learning_theory_lean` est importé **nativement** en tête de session (cache
d'oleans Mathlib construit une fois pour le poste). Les noms ci-dessous sont ceux du
fichier : les `section` de Lean 4 ne créent pas de namespace, donc `prop1_zeroLoss`
se cite sans préfixe.

Déclarations du module `EffectiveTheory.Grokking`, section `Parallelograms` :

In [1]:
import Mathlib
import EffectiveTheory.Grokking
import EffectiveTheory.GrokkingLemmas

open LearningTheory.EffectiveTheory

-- Definition 1 : le delta-parallelogramme
#check @IsDeltaParallelogram
-- Le cas delta = 0 : l'equivalence entre parallelogramme et egalite des sommes
#check @parallelogram_iff_eq
-- Proposition 1 : perte nulle + etiquettes injectives  =>  i + j = m + n
#check @prop1_zeroLoss
-- Proposition 2 : perte nulle + decodeur injectif  =>  le parallelogramme
#check @prop2_injectiveDecoder

import Mathlib
import EffectiveTheory.Grokking
import EffectiveTheory.GrokkingLemmas

open LearningTheory.EffectiveTheory

-- Definition 1 : le delta-parallelogramme
#check @IsDeltaParallelogram
──────▶  @IsDeltaParallelogram : {p : ℕ} →
  {V : Type u_1} → [NormedAddCommGroup V] → (Fin p → V) → ℝ → Fin p × Fin p → Fin p × Fin p → Prop
-- Le cas delta = 0 : l'equivalence entre parallelogramme et egalite des sommes
#check @parallelogram_iff_eq
──────▶  @parallelogram_iff_eq : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] (E : Fin p → V) (q r : Fin p × Fin p),
  IsDeltaParallelogram E 0 q r ↔ E q.1 + E q.2 = E r.1 + E r.2
-- Proposition 1 : perte nulle + etiquettes injectives  =>  i + j = m + n
#check @prop1_zeroLoss
──────▶  @prop1_zeroLoss : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] {Y : Type u_2} (E : Fin p → V) (dec : V → Y)
  (label : ℕ → Y),
  Function.Injective label →
    ∀ (D : Finset (Fin p × Fin p)),
      (∀ q ∈ D, dec (E q.1 + E q.2) = label (↑q.1 + ↑q.2)) →
        ∀ {q r : Fin p × Fin p}, q ∈ D → r ∈ D → E q.1 + E q.2 = E r.1 + E r.2 → ↑q.1 + ↑q.2 = ↑r.1 + ↑r.2
-- Proposition 2 : perte nulle + decodeur injectif  =>  le parallelogramme
#check @prop2_injectiveDecoder
──────▶  @prop2_injectiveDecoder : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] {Y : Type u_2} (E : Fin p → V)
  (dec : V → Y) (label : ℕ → Y) (D : Finset (Fin p × Fin p)),
  (∀ q ∈ D, dec (E q.1 + E q.2) = label (↑q.1 + ↑q.2)) →
    Function.Injective dec →
      ∀ {q r : Fin p × Fin p}, q ∈ D → r ∈ D → ↑q.1 + ↑q.2 = ↑r.1 + ↑r.2 → E q.1 + E q.2 = E r.1 + E r.2
--% env 0

Raw input:
{"cmd": "import Mathlib\nimport EffectiveTheory.Grokking\nimport EffectiveTheory.GrokkingLemmas\n\nopen LearningTheory.EffectiveTheory\n\n-- Definition 1 : le delta-parallelogramme\n#check @IsDeltaParallelogram\n-- Le cas delta = 0 : l'equivalence entre parallelogramme et egalite des sommes\n#check @parallelogram_iff_eq\n-- Proposition 1 : perte nulle + etiquettes injectives  =>  i + j = m + n\n#check @prop1_zeroLoss\n-- Proposition 2 : perte nulle + decodeur injectif  =>  le parallelogramme\n#check @prop2_injectiveDecoder"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@IsDeltaParallelogram : {p : ℕ} →\n  {V : Type u_1} → [NormedAddCommGroup V] → (Fin p → V) → ℝ → Fin p × Fin p → Fin p × Fin p → Prop"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "@parallelogram_iff_eq : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] (E : Fin p → V) (q r : Fin p × Fin p),\n  IsDeltaParallelogram E 0 q r ↔ E q.1 + E q.2 = E r.1 + E r.2"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data":
   "@prop1_zeroLoss : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] {Y : Type u_2} (E : Fin p → V) (dec : V → Y)\n  (label : ℕ → Y),\n  Function.Injective label →\n    ∀ (D : Finset (Fin p × Fin p)),\n      (∀ q ∈ D, dec (E q.1 + E q.2) = label (↑q.1 + ↑q.2)) →\n        ∀ {q r : Fin p × Fin p}, q ∈ D → r ∈ D → E q.1 + E q.2 = E r.1 + E r.2 → ↑q.1 + ↑q.2 = ↑r.1 + ↑r.2"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 6},
   "data":
   "@prop2_injectiveDecoder : ∀ {p : ℕ} {V : Type u_1} [inst : NormedAddCommGroup V] {Y : Type u_2} (E : Fin p → V)\n  (dec : V → Y) (label : ℕ → Y) (D : Finset (Fin p × Fin p)),\n  (∀ q ∈ D, dec (E q.1 + E q.2) = label (↑q.1 + ↑q.2)) →\n    Function.Injective dec →\n      ∀ {q r : Fin p × Fin p}, q ∈ D → r ∈ D → ↑q.1 + ↑q.2 = ↑r.1 + ↑r.2 → E q.1 + E q.2 = E r.1 + E r.2"}],
 "env": 0}

**Lecture du résultat.** Les quatre énoncés sont là, avec leurs hypothèses visibles :
`prop1_zeroLoss` prend une injection `Function.Injective label`, une perte nulle
`hZeroLoss`, et deux quadruplets du jeu d'entraînement `D` ; elle rend
`(q.1 : Nat) + q.2 = (r.1 : Nat) + r.2`. C'est exactement l'énoncé du papier : *à
perte nulle, un parallélogramme de la représentation est un fait arithmétique*.

Le champ `{V : Type*} [NormedAddCommGroup V]` dit que la preuve ne suppose ni la
dimension ni la nature de `V` : le résultat est indépendant de l'architecture, ce
qui est précisément ce qu'on attend d'une théorie *effective*.

## 2. Apprendre

La perte du papier est `loss0 P E = sum_{t in P} (linQ t E)^2`, où `linQ t E` est le
résidu de parallélogramme du quadruplet `t`. Dans le lake elle est déclarée
`noncomputable` : elle vit sur `Real` et sert à **prouver**, pas à **calculer**.

Pour exécuter la descente, on en écrit un **miroir calculable** sur `Float`, terme à
terme identique. Le miroir n'ajoute rien au lake : il refait la même somme, sur un
type où `#eval` sait descendre. Les théorèmes restent ceux du lake, ils sont
seulement mesurés sur les valeurs que ce miroir produit.

Le jeu d'entraînement `P5` est l'ensemble des quadruplets de paires d'indices de
`0..4` de **mêmes sommes** - c'est-à-dire, dans le papier, les quadruplets de paires
qui portent la même étiquette.

In [2]:
-- Miroir calculable de `loss0` : le residu de parallelogramme, puis sa somme
-- des carres. Meme formule que le lake (Eq. 5 et 24 du papier).
def linQF (E : Array Float) (t : (Nat × Nat) × (Nat × Nat)) : Float :=
  E[(t.1.1)]! + E[(t.1.2)]! - (E[(t.2.1)]! + E[(t.2.2)]!)

def loss0F (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Float :=
  P.foldl (fun acc t => acc + linQF E t * linQF E t) 0.0

-- Miroir du gradient `gradLoss0` : la derivee de chaque carre est
-- 2 * residu, portee avec le signe de la composante dans le residu.
def gradF (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Array Float :=
  Id.run do
    let mut g := Array.replicate E.size 0.0
    for t in P do
      let q := linQF E t
      g := g.set! t.1.1 (g[(t.1.1)]! + 2.0 * q)
      g := g.set! t.1.2 (g[(t.1.2)]! + 2.0 * q)
      g := g.set! t.2.1 (g[(t.2.1)]! - 2.0 * q)
      g := g.set! t.2.2 (g[(t.2.2)]! - 2.0 * q)
    return g

def pasDeDescente (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float)
    (eta : Float) : Array Float :=
  Id.run do
    let g := gradF P E
    let mut E2 := E
    for i in List.range E.size do
      E2 := E2.set! i (E[i]! - eta * g[i]!)
    return E2

def entrainer (P : List ((Nat × Nat) × (Nat × Nat))) (E0 : Array Float)
    (eta : Float) (pas : Nat) : Array Float :=
  Id.run do
    let mut E := E0
    for _ in List.range pas do
      E := pasDeDescente P E eta
    return E

-- Les huit quadruplets de paires de 0..4 de memes sommes (donc de memes etiquettes).
def P5 : List ((Nat × Nat) × (Nat × Nat)) :=
  [((0, 1), (1, 0)), ((0, 2), (1, 1)), ((0, 3), (1, 2)), ((1, 2), (2, 1)),
   ((1, 3), (2, 2)), ((2, 3), (3, 2)), ((0, 4), (2, 2)), ((1, 4), (2, 3))]

-- Un embedding initial quelconque : rien n'y est range.
def E0 : Array Float := #[0.3, -1.2, 0.7, 2.1, -0.4]

-- La perte avant, puis apres 200 pas de descente (eta = 0,05).
#eval loss0F P5 E0
#eval loss0F P5 (entrainer P5 E0 0.05 200)

-- Miroir calculable de `loss0` : le residu de parallelogramme, puis sa somme
-- des carres. Meme formule que le lake (Eq. 5 et 24 du papier).
def linQF (E : Array Float) (t : (Nat × Nat) × (Nat × Nat)) : Float :=
  E[(t.1.1)]! + E[(t.1.2)]! - (E[(t.2.1)]! + E[(t.2.2)]!)

def loss0F (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Float :=
  P.foldl (fun acc t => acc + linQF E t * linQF E t) 0.0

-- Miroir du gradient `gradLoss0` : la derivee de chaque carre est
-- 2 * residu, portee avec le signe de la composante dans le residu.
def gradF (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Array Float :=
  Id.run do
    let mut g := Array.replicate E.size 0.0
    for t in P do
      let q := linQF E t
      g := g.set! t.1.1 (g[(t.1.1)]! + 2.0 * q)
      g := g.set! t.1.2 (g[(t.1.2)]! + 2.0 * q)
      g := g.set! t.2.1 (g[(t.2.1)]! - 2.0 * q)
      g := g.set! t.2.2 (g[(t.2.2)]! - 2.0 * q)
    return g

def pasDeDescente (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float)
    (eta : Float) : Array Float :=
  Id.run do
    let g := gradF P E
    let mut E2 := E
    for i in List.range E.size do
      E2 := E2.set! i (E[i]! - eta * g[i]!)
    return E2

def entrainer (P : List ((Nat × Nat) × (Nat × Nat))) (E0 : Array Float)
    (eta : Float) (pas : Nat) : Array Float :=
  Id.run do
    let mut E := E0
    for _ in List.range pas do
      E := pasDeDescente P E eta
    return E

-- Les huit quadruplets de paires de 0..4 de memes sommes (donc de memes etiquettes).
def P5 : List ((Nat × Nat) × (Nat × Nat)) :=
  [((0, 1), (1, 0)), ((0, 2), (1, 1)), ((0, 3), (1, 2)), ((1, 2), (2, 1)),
   ((1, 3), (2, 2)), ((2, 3), (3, 2)), ((0, 4), (2, 2)), ((1, 4), (2, 3))]

-- Un embedding initial quelconque : rien n'y est range.
def E0 : Array Float := #[0.3, -1.2, 0.7, 2.1, -0.4]

-- La perte avant, puis apres 200 pas de descente (eta = 0,05).
#eval loss0F P5 E0
─────▶  41.830000
#eval loss0F P5 (entrainer P5 E0 0.05 200)
─────▶  0.000000
--% env 1

Raw input:
{"cmd": "-- Miroir calculable de `loss0` : le residu de parallelogramme, puis sa somme\n-- des carres. Meme formule que le lake (Eq. 5 et 24 du papier).\ndef linQF (E : Array Float) (t : (Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat)) : Float :=\n  E[(t.1.1)]! + E[(t.1.2)]! - (E[(t.2.1)]! + E[(t.2.2)]!)\n\ndef loss0F (P : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat))) (E : Array Float) : Float :=\n  P.foldl (fun acc t => acc + linQF E t * linQF E t) 0.0\n\n-- Miroir du gradient `gradLoss0` : la derivee de chaque carre est\n-- 2 * residu, portee avec le signe de la composante dans le residu.\ndef gradF (P : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat))) (E : Array Float) : Array Float :=\n  Id.run do\n    let mut g := Array.replicate E.size 0.0\n    for t in P do\n      let q := linQF E t\n      g := g.set! t.1.1 (g[(t.1.1)]! + 2.0 * q)\n      g := g.set! t.1.2 (g[(t.1.2)]! + 2.0 * q)\n      g := g.set! t.2.1 (g[(t.2.1)]! - 2.0 * q)\n      g := g.set! t.2.2 (g[(t.2.2)]! - 2.0 * q)\n    return g\n\ndef pasDeDescente (P : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat))) (E : Array Float)\n    (eta : Float) : Array Float :=\n  Id.run do\n    let g := gradF P E\n    let mut E2 := E\n    for i in List.range E.size do\n      E2 := E2.set! i (E[i]! - eta * g[i]!)\n    return E2\n\ndef entrainer (P : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat))) (E0 : Array Float)\n    (eta : Float) (pas : Nat) : Array Float :=\n  Id.run do\n    let mut E := E0\n    for _ in List.range pas do\n      E := pasDeDescente P E eta\n    return E\n\n-- Les huit quadruplets de paires de 0..4 de memes sommes (donc de memes etiquettes).\ndef P5 : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat)) :=\n  [((0, 1), (1, 0)), ((0, 2), (1, 1)), ((0, 3), (1, 2)), ((1, 2), (2, 1)),\n   ((1, 3), (2, 2)), ((2, 3), (3, 2)), ((0, 4), (2, 2)), ((1, 4), (2, 3))]\n\n-- Un embedding initial quelconque : rien n'y est range.\ndef E0 : Array Float := #[0.3, -1.2, 0.7, 2.1, -0.4]\n\n-- La perte avant, puis apres 200 pas de d

**Lecture du résultat.** La perte part d'une valeur non nulle et tombe à l'ordre du
bruit numérique : le miroir descend bien le même objectif que le lake étudie.

Cette descente est le **jouet** du papier, pas un réseau de neurones : c'est le
point important. R02 ne prouve pas qu'un perceptron grokke ; il prouve que *si* la
perte d'entraînement s'annule *et* que les étiquettes sont injectives, *alors* la
géométrie de la représentation est contrainte. La section 2 réalise l'hypothèse,
la section 4 en tire la conclusion - formellement.

## 3. Extraire

Ce que l'expérience ci-dessus produit, c'est un **invariant** : l'embedding appris
n'est pas n'importe quel vecteur, il vérifie une relation. On mesure ici le résidu
`linQ` quadruplet par quadruplet, après entraînement.

Les huit quadruplets de `P5` sont ceux dont les sommes coïncident. Le **contre-témoin**
est un quadruplet dont les sommes diffèrent : `((0, 0), (2, 3))`, soit `0` contre `5`.
S'il ne se comporte pas comme les autres, l'invariant n'est pas « le résidu est petit
partout » - il est « le résidu est petit **exactement là où** l'arithmétique le dit ».

In [3]:
-- Residus des huit quadruplets d'entrainement, apres apprentissage.
#eval P5.map (fun t => linQF (entrainer P5 E0 0.05 200) t)

-- Contre-temoin : un quadruplet HORS de P5 (sommes 0 et 5, differentes).
#eval linQF (entrainer P5 E0 0.05 200) ((0, 0), (2, 3))

-- Residus des huit quadruplets d'entrainement, apres apprentissage.
#eval P5.map (fun t => linQF (entrainer P5 E0 0.05 200) t)
─────▶  [0.000000, -0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000]

-- Contre-temoin : un quadruplet HORS de P5 (sommes 0 et 5, differentes).
#eval linQF (entrainer P5 E0 0.05 200) ((0, 0), (2, 3))
─────▶  -0.950000
--% env 2

Raw input:
{"cmd": "-- Residus des huit quadruplets d'entrainement, apres apprentissage.\n#eval P5.map (fun t => linQF (entrainer P5 E0 0.05 200) t)\n\n-- Contre-temoin : un quadruplet HORS de P5 (sommes 0 et 5, differentes).\n#eval linQF (entrainer P5 E0 0.05 200) ((0, 0), (2, 3))", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[0.000000, -0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "-0.950000"}],
 "env": 2}

**Lecture du résultat.** Les huit résidus d'entraînement sont à l'ordre du bruit,
le contre-témoin ne l'est pas : l'invariant extrait est bien **sélectif**. La
représentation a été rangée par l'arithmétique de la tâche, pas aplatie.

C'est le fait que la Proposition 2 explique : un décodeur injectif **interdit** à
deux sommes différentes de partager une représentation. La section 4 énonce ce fait
et le premier, sous forme de théorèmes.

## 4. Prouver

Les théorèmes existent ; encore faut-il qu'ils soient **utilisables**. Un `#check`
qui affiche un type ne prouve rien : il montre une signature. On instancie donc ici
les deux propositions sur des objets concrets, et on demande au noyau quels axiomes
la preuve de la Proposition 1 a effectivement consommés.

Le dernier point est le contrôle d'intégrité : une preuve qui trahirait un `sorry`
ou un `native_decide` le dirait ici.

Attention à l'arité de `IsDeltaParallelogram` : elle prend **deux** paires
`(q r : Fin p × Fin p)`, pas une paire de paires. La forme `(0, 1) (1, 0)` est la
bonne ; `((0, 1), (1, 0))` serait un seul argument et le noyau répondrait
`type expected`.

In [4]:
-- Proposition 1, instanciee : le theoreme EST applicable, pas seulement affichable.
example (E : Fin 3 → ℝ) (dec : ℝ → ℕ) (label : ℕ → ℕ)
    (hLabel : Function.Injective label) (D : Finset (Fin 3 × Fin 3))
    (hZero : ∀ q ∈ D, dec (E q.1 + E q.2) = label ((q.1 : ℕ) + q.2))
    {q r : Fin 3 × Fin 3} (hq : q ∈ D) (hr : r ∈ D)
    (hPara : E q.1 + E q.2 = E r.1 + E r.2) :
    (q.1 : ℕ) + q.2 = (r.1 : ℕ) + r.2 :=
  prop1_zeroLoss E dec label hLabel D hZero hq hr hPara

-- L'equivalence du lake, prise dans le sens « parallelogramme => egalite ».
example (E : Fin 3 → ℝ) (h : E 0 + E 1 = E 1 + E 0) :
    IsDeltaParallelogram E 0 (0, 1) (1, 0) := by
  rw [parallelogram_iff_eq]
  exact h

-- Controle d'integrite : quels axiomes la Proposition 1 consomme-t-elle ?
#print axioms prop1_zeroLoss

-- Proposition 1, instanciee : le theoreme EST applicable, pas seulement affichable.
example (E : Fin 3 → ℝ) (dec : ℝ → ℕ) (label : ℕ → ℕ)
    (hLabel : Function.Injective label) (D : Finset (Fin 3 × Fin 3))
    (hZero : ∀ q ∈ D, dec (E q.1 + E q.2) = label ((q.1 : ℕ) + q.2))
    {q r : Fin 3 × Fin 3} (hq : q ∈ D) (hr : r ∈ D)
    (hPara : E q.1 + E q.2 = E r.1 + E r.2) :
    (q.1 : ℕ) + q.2 = (r.1 : ℕ) + r.2 :=
  prop1_zeroLoss E dec label hLabel D hZero hq hr hPara

-- L'equivalence du lake, prise dans le sens « parallelogramme => egalite ».
example (E : Fin 3 → ℝ) (h : E 0 + E 1 = E 1 + E 0) :
    IsDeltaParallelogram E 0 (0, 1) (1, 0) := by
  rw [parallelogram_iff_eq]
  exact h

-- Controle d'integrite : quels axiomes la Proposition 1 consomme-t-elle ?
#print axioms prop1_zeroLoss
──────▶  'LearningTheory.EffectiveTheory.prop1_zeroLoss' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 3

Raw input:
{"cmd": "-- Proposition 1, instanciee : le theoreme EST applicable, pas seulement affichable.\nexample (E : Fin 3 \u2192 \u211d) (dec : \u211d \u2192 \u2115) (label : \u2115 \u2192 \u2115)\n    (hLabel : Function.Injective label) (D : Finset (Fin 3 \u00d7 Fin 3))\n    (hZero : \u2200 q \u2208 D, dec (E q.1 + E q.2) = label ((q.1 : \u2115) + q.2))\n    {q r : Fin 3 \u00d7 Fin 3} (hq : q \u2208 D) (hr : r \u2208 D)\n    (hPara : E q.1 + E q.2 = E r.1 + E r.2) :\n    (q.1 : \u2115) + q.2 = (r.1 : \u2115) + r.2 :=\n  prop1_zeroLoss E dec label hLabel D hZero hq hr hPara\n\n-- L'equivalence du lake, prise dans le sens \u00ab parallelogramme => egalite \u00bb.\nexample (E : Fin 3 \u2192 \u211d) (h : E 0 + E 1 = E 1 + E 0) :\n    IsDeltaParallelogram E 0 (0, 1) (1, 0) := by\n  rw [parallelogram_iff_eq]\n  exact h\n\n-- Controle d'integrite : quels axiomes la Proposition 1 consomme-t-elle ?\n#print axioms prop1_zeroLoss", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 6},
   "data":
   "'LearningTheory.EffectiveTheory.prop1_zeroLoss' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 3}

**Lecture du résultat.** `#print axioms` rend la liste des axiomes sur lesquels la
preuve repose. Elle ne doit contenir **ni** `sorryAx` **ni** `native_decide.*` : le
premier signalerait un trou de preuve, le second un théorème vide par réduction non
prouvée. Ce carnet n'ajoute aucun `sorry` à un énoncé du lake ; les seuls `sorry`
sont dans les exercices de la section 7, où ils sont l'énoncé lui-même.

Les axiomes qui restent sont ceux du socle classique de Lean 4 - `propext`,
`Quot.sound` et `Classical.choice` (le choix non constructif, que Mathlib utilise
partout). C'est la signature attendue d'une preuve ordinaire : les lire, c'est
constater qu'il n'y a rien **d'autre**.

## 5. Conserver

L'Appendice F du papier ne s'arrête pas aux parallélogrammes : il établit que la
descente de gradient de la perte **effective** `l_eff = l0 / Z0` conserve deux
quantités - `Z0 = sum_k (E k)^2` (Eq. 26) et, à perte nulle, `C = sum_k E k` (Eq. 27).
C'est ce qui interdit à la représentation de s'effondrer à zéro : elle se réorganise,
elle ne disparaît pas.

Le lake porte ces énoncés dans la section `Flow`. On en vérifie ici la pièce
calculatoire, l'**identité d'Euler** `sum_k (d l0 / d E_k) * E_k = 2 * l0` - une
fonction quadratique homogène de degré 2 - sur l'embedding effectivement entraîné.
C'est le pont entre la mesure et la preuve : ce que la section 2 a calculé doit
vérifier ce que la section 5 affirme.

In [5]:
-- Les enonces de conservation du lake.
#check @loss0_grad_sum_zero
#check @loss0_grad_dot_self
#check @flow_sumsq0_constant
#check @flow_sum_constant_of_zero_loss

-- Temoin numerique de l'identite d'Euler : sum_k (d l0 / d E_k) * E_k = 2 * l0.
def eulerGauche (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Float :=
  Id.run do
    let g := gradF P E
    let mut s := 0.0
    for i in List.range E.size do
      s := s + g[i]! * E[i]!
    return s

-- L'identite est algebrique : on la mesure a TROIS points, dont deux ou la perte
-- est non nulle. Au point entraine elle se reduit a 0 = 0 et ne discrimine rien.
#eval (eulerGauche P5 E0, 2.0 * loss0F P5 E0)
#eval (eulerGauche P5 (entrainer P5 E0 0.05 20), 2.0 * loss0F P5 (entrainer P5 E0 0.05 20))
#eval (eulerGauche P5 (entrainer P5 E0 0.05 200), 2.0 * loss0F P5 (entrainer P5 E0 0.05 200))

-- Les enonces de conservation du lake.
#check @loss0_grad_sum_zero
──────▶  @loss0_grad_sum_zero : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) (E : Fin p → ℝ),
  ∑ k, (fderiv ℝ (loss0 P) E) (Pi.single k 1) = 0
#check @loss0_grad_dot_self
──────▶  @loss0_grad_dot_self : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) (E : Fin p → ℝ),
  ∑ k, (fderiv ℝ (loss0 P) E) (Pi.single k 1) * E k = 2 * loss0 P E
#check @flow_sumsq0_constant
──────▶  @flow_sumsq0_constant : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) {γ : ℝ → Fin p → ℝ},
  IsEffectiveFlow P γ → ∀ (s t : ℝ), sumsq0 (γ s) = sumsq0 (γ t)
#check @flow_sum_constant_of_zero_loss
──────▶  @flow_sum_constant_of_zero_loss : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) {γ : ℝ → Fin p → ℝ},
  IsEffectiveFlow P γ → (∀ (s : ℝ), loss0 P (γ s) = 0) → ∀ (s t : ℝ), ∑ k, γ s k = ∑ k, γ t k

-- Temoin numerique de l'identite d'Euler : sum_k (d l0 / d E_k) * E_k = 2 * l0.
def eulerGauche (P : List ((Nat × Nat) × (Nat × Nat))) (E : Array Float) : Float :=
  Id.run do
    let g := gradF P E
    let mut s := 0.0
    for i in List.range E.size do
      s := s + g[i]! * E[i]!
    return s

-- L'identite est algebrique : on la mesure a TROIS points, dont deux ou la perte
-- est non nulle. Au point entraine elle se reduit a 0 = 0 et ne discrimine rien.
#eval (eulerGauche P5 E0, 2.0 * loss0F P5 E0)
─────▶  (83.660000, 83.660000)
#eval (eulerGauche P5 (entrainer P5 E0 0.05 20), 2.0 * loss0F P5 (entrainer P5 E0 0.05 20))
─────▶  (0.000001, 0.000001)
#eval (eulerGauche P5 (entrainer P5 E0 0.05 200), 2.0 * loss0F P5 (entrainer P5 E0 0.05 200))
─────▶  (0.000000, 0.000000)
--% env 4

Raw input:
{"cmd": "-- Les enonces de conservation du lake.\n#check @loss0_grad_sum_zero\n#check @loss0_grad_dot_self\n#check @flow_sumsq0_constant\n#check @flow_sum_constant_of_zero_loss\n\n-- Temoin numerique de l'identite d'Euler : sum_k (d l0 / d E_k) * E_k = 2 * l0.\ndef eulerGauche (P : List ((Nat \u00d7 Nat) \u00d7 (Nat \u00d7 Nat))) (E : Array Float) : Float :=\n  Id.run do\n    let g := gradF P E\n    let mut s := 0.0\n    for i in List.range E.size do\n      s := s + g[i]! * E[i]!\n    return s\n\n-- L'identite est algebrique : on la mesure a TROIS points, dont deux ou la perte\n-- est non nulle. Au point entraine elle se reduit a 0 = 0 et ne discrimine rien.\n#eval (eulerGauche P5 E0, 2.0 * loss0F P5 E0)\n#eval (eulerGauche P5 (entrainer P5 E0 0.05 20), 2.0 * loss0F P5 (entrainer P5 E0 0.05 20))\n#eval (eulerGauche P5 (entrainer P5 E0 0.05 200), 2.0 * loss0F P5 (entrainer P5 E0 0.05 200))", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@loss0_grad_sum_zero : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) (E : Fin p → ℝ),\n  ∑ k, (fderiv ℝ (loss0 P) E) (Pi.single k 1) = 0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@loss0_grad_dot_self : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) (E : Fin p → ℝ),\n  ∑ k, (fderiv ℝ (loss0 P) E) (Pi.single k 1) * E k = 2 * loss0 P E"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@flow_sumsq0_constant : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) {γ : ℝ → Fin p → ℝ},\n  IsEffectiveFlow P γ → ∀ (s t : ℝ), sumsq0 (γ s) = sumsq0 (γ t)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@flow_sum_constant_of_zero_loss : ∀ {p : ℕ} (P : Finset ((Fin p × Fin p) × Fin p × Fin p)) {γ : ℝ → Fin p → ℝ},\n  IsEffectiveFlow P γ → (∀ (s : ℝ), loss0 P (γ s) = 0) → ∀ (s t : ℝ), ∑ k, γ s k = ∑ k, γ t k"},
  {"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 5},
   "data": "(83.660000, 83.660000)"},
  {"severity": "info",
   "pos": {"line": 19, "column

**Lecture du résultat.** Les deux colonnes coïncident aux **trois** points, dont deux
où la perte est **non nulle** : l'identité que le lake **prouve** est celle que le
miroir **mesure**.

La mesure est faite à trois instants à dessein. Au point entraîné, la perte est
nulle : l'identité s'y réduit à `0 = 0` et **ne discrimine rien**. Une mesure qui ne
peut pas échouer ne prouve pas qu'elle tient - c'est précisément pour cela que les
deux premiers points sont mesurés sur un embedding où le gradient travaille encore.
C'est tout l'intérêt de faire cohabiter preuve et mesure : une preuve sans mesure ne
dit pas que le modèle étudié est dans son domaine, une mesure sans preuve ne dit pas
pourquoi elle tient.

Ces trois points ne sont pas une courbe intégrale du flot effectif (la section 2
descend `l0`, pas `l0 / Z0`), et ce n'est pas nécessaire : l'identité d'Euler est
algébrique - elle vaut pour tout `E` -, ce qui est exactement ce que dit l'énoncé de
`loss0_grad_dot_self`, qui ne porte aucune hypothèse de flot.

## 6. Limites

Écrites ici plutôt que découvertes plus tard.

- **Le carnet ne rejoue pas l'apprentissage réel.** Aucun réseau de neurones n'est
  entraîné ici : la section 2 descend la perte effective du papier sur un tableau
  d'embeddings de cinq entrées. Le grokking d'un perceptron est mesuré dans
  [2.9](2.9-Grokking-Generalisation.ipynb) et [2.9c](2.9c-Grokking-Diagrammes-Phases.ipynb),
  sur PyTorch ; les deux ne se remplacent pas.
- **Le miroir `Float` n'est pas le lake.** `loss0F` refait la somme de `loss0` en
  virgule flottante ; les théorèmes, eux, portent sur `Real`. Le miroir ne prouve
  rien et ne doit pas être cité comme preuve - il est là pour que la descente soit
  exécutable.
- **Le régime approché n'est pas couvert.** Les propositions du lake supposent la
  perte d'entraînement **exactement** nulle. Une perte petite mais non nulle ne les
  atteint pas ; c'est `IsDeltaParallelogram` avec `delta > 0` qui décrit ce régime,
  et le lake n'en tire pas la conclusion arithmétique - c'est l'exercice 1 qui en
  mesure la seule conséquence immédiate (la monotonie en `delta`).
- **L'optimiseur n'est pas le sujet** : le lake ne dit rien de la descente, ni du
  nombre de pas, ni du taux d'apprentissage. La théorie est *effective* - elle
  décrit ce que la représentation doit vérifier à l'équilibre, pas comment y arriver.
- **La présentation n'est pas la première du dépôt.** Le carnet `SL-1b` de la série
  `SymbolicAI/SymbolicLearning` (§8) présente déjà ces mêmes déclarations, par
  `#check`. Ce carnet ne les remplace pas : il les instancie et les mesure.

## 7. Exercices

Trois énoncés à compléter. Le corps de ce carnet ne contient aucun `sorry` en dehors
de ces trois cellules : ils sont l'exercice, pas un raccourci.

Ils suivent les questions ouvertes du volet A.2.3 du corpus : *quelle est la marge
réelle d'un énoncé approché ? que protège exactement une hypothèse d'injectivité ?
sur quoi repose une quantité conservée ?*

In [6]:
-- Exercice 1 : la monotonie en delta.
-- TODO etudiant : completer.
-- Indice : `IsDeltaParallelogram` est une INEGALITE (`‖...‖ ≤ delta`) ;
-- elargir delta ne peut que l'affaiblir. `exact le_trans h hdelta` suffit.
example (E : Fin 3 → ℝ) {delta delta' : ℝ} (hdelta : delta ≤ delta')
    (q r : Fin 3 × Fin 3) (h : IsDeltaParallelogram E delta q r) :
    IsDeltaParallelogram E delta' q r := by
  sorry

-- Exercice 1 : la monotonie en delta.
-- TODO etudiant : completer.
-- Indice : `IsDeltaParallelogram` est une INEGALITE (`‖...‖ ≤ delta`) ;
-- elargir delta ne peut que l'affaiblir. `exact le_trans h hdelta` suffit.
example (E : Fin 3 → ℝ) {delta delta' : ℝ} (hdelta : delta ≤ delta')
───────▶ 🟨 declaration uses `sorry`
    (q r : Fin 3 × Fin 3) (h : IsDeltaParallelogram E delta q r) :
    IsDeltaParallelogram E delta' q r := by
  sorry
--% env 5
--% prove 0

Raw input:
{"cmd": "-- Exercice 1 : la monotonie en delta.\n-- TODO etudiant : completer.\n-- Indice : `IsDeltaParallelogram` est une INEGALITE (`\u2016...\u2016 \u2264 delta`) ;\n-- elargir delta ne peut que l'affaiblir. `exact le_trans h hdelta` suffit.\nexample (E : Fin 3 \u2192 \u211d) {delta delta' : \u211d} (hdelta : delta \u2264 delta')\n    (q r : Fin 3 \u00d7 Fin 3) (h : IsDeltaParallelogram E delta q r) :\n    IsDeltaParallelogram E delta' q r := by\n  sorry", "env": 4}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 8, "column": 2},
   "goal":
   "E : Fin 3 → ℝ\ndelta delta' : ℝ\nhdelta : delta ≤ delta'\nq r : Fin 3 × Fin 3\nh : IsDeltaParallelogram E delta q r\n⊢ IsDeltaParallelogram E delta' q r",
   "endPos": {"line": 8, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 7},
   "data": "declaration uses `sorry`"}],
 "env": 5}

In [7]:
-- Exercice 2 : ce que protege l'injectivite.
-- TODO etudiant : completer.
-- Indice : l'hypothese `hLabel` de `prop1_zeroLoss` ne peut pas etre retiree.
-- Commencer par exhiber deux indices distincts de meme etiquette sous la fonction
-- constante `fun _ : Nat => 0`. `exact ⟨0, 1, by decide, rfl⟩` suffit.
example : ∃ a b : ℕ, a ≠ b ∧ (fun _ : ℕ => (0 : ℕ)) a = (fun _ : ℕ => (0 : ℕ)) b := by
  sorry

-- Exercice 2 : ce que protege l'injectivite.
-- TODO etudiant : completer.
-- Indice : l'hypothese `hLabel` de `prop1_zeroLoss` ne peut pas etre retiree.
-- Commencer par exhiber deux indices distincts de meme etiquette sous la fonction
-- constante `fun _ : Nat => 0`. `exact ⟨0, 1, by decide, rfl⟩` suffit.
example : ∃ a b : ℕ, a ≠ b ∧ (fun _ : ℕ => (0 : ℕ)) a = (fun _ : ℕ => (0 : ℕ)) b := by
───────▶ 🟨 declaration uses `sorry`
  sorry
--% env 6
--% prove 1

Raw input:
{"cmd": "-- Exercice 2 : ce que protege l'injectivite.\n-- TODO etudiant : completer.\n-- Indice : l'hypothese `hLabel` de `prop1_zeroLoss` ne peut pas etre retiree.\n-- Commencer par exhiber deux indices distincts de meme etiquette sous la fonction\n-- constante `fun _ : Nat => 0`. `exact \u27e80, 1, by decide, rfl\u27e9` suffit.\nexample : \u2203 a b : \u2115, a \u2260 b \u2227 (fun _ : \u2115 => (0 : \u2115)) a = (fun _ : \u2115 => (0 : \u2115)) b := by\n  sorry", "env": 5}
Raw output:
{"sorries":
 [{"proofState": 1,
   "pos": {"line": 7, "column": 2},
   "goal": "⊢ ∃ a b, a ≠ b ∧ (fun x => 0) a = (fun x => 0) b",
   "endPos": {"line": 7, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 7},
   "data": "declaration uses `sorry`"}],
 "env": 6}

In [8]:
-- Exercice 3 : l'energie ne peut pas etre negative.
-- TODO etudiant : completer.
-- Indice : `sumsq0 E = sum_k (E k)^2` ; chaque terme est un carre, donc positif.
-- `Finset.sum_nonneg` et `sq_nonneg` font le travail. Noter que `sumsq0` est un
-- objet du LAKE - l'exercice porte sur le lake, pas sur une copie locale.
example (E : Fin 3 → ℝ) : 0 ≤ sumsq0 E := by
  sorry

-- Exercice 3 : l'energie ne peut pas etre negative.
-- TODO etudiant : completer.
-- Indice : `sumsq0 E = sum_k (E k)^2` ; chaque terme est un carre, donc positif.
-- `Finset.sum_nonneg` et `sq_nonneg` font le travail. Noter que `sumsq0` est un
-- objet du LAKE - l'exercice porte sur le lake, pas sur une copie locale.
example (E : Fin 3 → ℝ) : 0 ≤ sumsq0 E := by
───────▶ 🟨 declaration uses `sorry`
  sorry
--% env 7
--% prove 2

Raw input:
{"cmd": "-- Exercice 3 : l'energie ne peut pas etre negative.\n-- TODO etudiant : completer.\n-- Indice : `sumsq0 E = sum_k (E k)^2` ; chaque terme est un carre, donc positif.\n-- `Finset.sum_nonneg` et `sq_nonneg` font le travail. Noter que `sumsq0` est un\n-- objet du LAKE - l'exercice porte sur le lake, pas sur une copie locale.\nexample (E : Fin 3 \u2192 \u211d) : 0 \u2264 sumsq0 E := by\n  sorry", "env": 6}
Raw output:
{"sorries":
 [{"proofState": 2,
   "pos": {"line": 7, "column": 2},
   "goal": "E : Fin 3 → ℝ\n⊢ 0 ≤ sumsq0 E",
   "endPos": {"line": 7, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 7},
   "data": "declaration uses `sorry`"}],
 "env": 7}

**Ce que ce carnet laisse derrière lui.** Trois énoncés du lake cessent d'être des
noms : `prop1_zeroLoss` et `prop2_injectiveDecoder` sont instanciés et leur intégrité
d'axiomes est lue ; les lois de conservation de l'App. F sont mesurées sur un
embedding réellement entraîné. La chaîne de la famille 2.9 est complète :
**apprendre** (2.9, 2.9c) -> **extraire** (2.9d, 2.9e) -> **prouver** (ici).

**Suite naturelle** : les modules voisins du même lake n'ont pas encore de compagnon
- `EffectiveTheory/CircleOfDays.lean` (R10, le cercle des jours comme représentation
irréductible de `C7`, dont [2.9d](2.9d-Features-Circulaires-Helice-Nombres.ipynb)
mesure la version apprise), `InfoBits.lean` et `Repons.lean` (contenu informationnel
`b = log2(n! / |Aut G|)`, la quantité conservée de la théorie effective).